# Ontology - Knowledge Graph

Builds the **OneGrid knowledge graph** as two Delta tables in the gold schema:

- `gold.ontology_nodes` - one row per entity (table) with category, grain, description and its columns (JSON).
- `gold.ontology_edges` - one row per relationship (physical / logical / temporal) with a human verb.

This mirrors the report app's **Ontology** screen, so the graph exists both locally (report-app/server/ontology.json) and in Fabric. 
Attach **lh_poc** as the default lakehouse before running.


In [ ]:
# PARAMETERS  (Fabric: this cell is tagged 'parameters')
LAKEHOUSE_SCHEMA = "gold"          # schema holding dim_/fact_ tables
OUT_SCHEMA       = "gold"          # where ontology_nodes / ontology_edges are written
NODES_TABLE      = "ontology_nodes"
EDGES_TABLE      = "ontology_edges"

In [ ]:
# ---- ontology definitions (kept in sync with _gen-ontology.py) ----
CATEGORIES = {
    "dimension": {"label": "Dimension", "color": "#5aa9ff"},
    "bridge":    {"label": "Bridge",    "color": "#a78bfa"},
    "telemetry": {"label": "Telemetry", "color": "#37e0d0"},
    "event":     {"label": "Events",    "color": "#ffcc4d"},
    "ml":        {"label": "ML Scoring","color": "#ff8c42"},
    "advisory":  {"label": "Advisory",  "color": "#ff5470"},
    "narrative": {"label": "Narrative", "color": "#8ea3bd"},
}

META = {
    "dim_asset": dict(label="Asset", category="dimension", role="hub", grain="one row per monitored asset",
        desc="The central entity: every generating asset (turbine, boiler, pump, generator) with plant, unit and category."),
    "dim_equipment": dict(label="Equipment", category="dimension", grain="one row per equipment item (iCare id)",
        desc="Physical equipment master keyed by iCare id - bridges condition-monitoring identity and analytics asset id."),
    "dim_date": dict(label="Date", category="dimension", grain="one row per calendar day",
        desc="Standard date dimension used to trend facts and scores over time."),
    "bridge_pi_tag_to_asset": dict(label="Tag to Asset Bridge", category="bridge", role="hub", grain="one row per PI tag",
        desc="Maps each PI sensor tag to the asset/equipment it belongs to."),
    "selected_tags": dict(label="Selected Tags", category="bridge", grain="one row per model-selected tag",
        desc="The curated set of tags chosen as features for the predictive models."),
    "fact_pi": dict(label="PI Telemetry", category="telemetry", grain="one row per tag per timestamp",
        desc="Raw process-historian (PI) time-series readings streamed from plant sensors."),
    "fact_icare_measurement": dict(label="iCare Measurements", category="telemetry", grain="one row per condition reading",
        desc="Condition-monitoring measurements (vibration, oil, thermography) from equipment rounds."),
    "aakr_scores": dict(label="AAKR Residuals", category="telemetry", grain="one row per tag per scoring run",
        desc="Auto-Associative Kernel Regression residuals - drift of each tag from its expected healthy value."),
    "aakr_health": dict(label="AAKR Health", category="telemetry", grain="one row per asset per scoring run",
        desc="Rolled-up asset health index derived from AAKR residuals."),
    "fact_gads_event": dict(label="GADS Outages", category="event", grain="one row per outage/derate event",
        desc="GADS reliability events - forced outages, derates and causes per asset."),
    "fact_work_requests": dict(label="Work Requests", category="event", grain="one row per work request",
        desc="Maintenance work requests / orders raised against equipment."),
    "predictions_longterm": dict(label="Long-Term Survival", category="ml", grain="one row per asset per horizon",
        desc="Cox survival output - 7/14-day survival probability, risk score, median days to failure."),
    "predictions_shortterm": dict(label="Short-Term Stop Risk", category="ml", grain="one row per asset per horizon",
        desc="Near-term stop-probability model - trip likelihood within 4h/8h/24h with an alert level."),
    "anomaly_advisories": dict(label="Anomaly Advisories", category="advisory", grain="one row per anomaly episode",
        desc="SmartSignal anomaly episodes per tag - peak z, direction, duration and advisory message."),
    "watchlist": dict(label="Watchlist", category="advisory", grain="one row per watched signal",
        desc="Ranked watch signals per asset - descriptor, normal range, trend and recommended action."),
    "root_cause": dict(label="Root Cause", category="advisory", grain="one row per diagnosed fault",
        desc="Diagnosed failure mechanisms per asset/tag with likely cause, confidence and recommended action."),
    "daily_narrative": dict(label="Daily Narrative", category="narrative", grain="one row per day",
        desc="Auto-generated plain-language daily briefing summarizing fleet/asset state."),
}

# physical relationships (mirrors the semantic model relationships.tmdl)
PHYSICAL_RELS = [
    ("predictions_longterm","asset_id","dim_asset","asset_id","long-term survival for"),
    ("fact_gads_event","asset_id","dim_asset","asset_id","outage events for"),
    ("fact_icare_measurement","asset_id","dim_asset","asset_id","condition readings for"),
    ("watchlist","asset_id","dim_asset","asset_id","watch signals for"),
    ("predictions_shortterm","asset_id","dim_asset","asset_id","short-term stop risk for"),
    ("dim_equipment","asset_id","dim_asset","asset_id","describes"),
    ("fact_pi","Tag","bridge_pi_tag_to_asset","Tag","telemetry on tag"),
    ("anomaly_advisories","Tag","bridge_pi_tag_to_asset","Tag","anomaly on tag"),
    ("aakr_health","asset_id","dim_asset","asset_id","health index for"),
    ("aakr_scores","tag","bridge_pi_tag_to_asset","Tag","residual on tag"),
    ("bridge_pi_tag_to_asset","asset_id","dim_equipment","icare_id","maps tag to"),
    ("fact_work_requests","entity_identity","dim_equipment","icare_id","work on"),
]
PK = {"dim_asset":"asset_id","dim_equipment":"icare_id","bridge_pi_tag_to_asset":"Tag"}

In [ ]:
import json
from datetime import datetime, timezone
from pyspark.sql import Row

def table_columns(t):
    for name in (f"{LAKEHOUSE_SCHEMA}.{t}", t):
        try:
            return [(f.name, f.dataType.simpleString()) for f in spark.table(name).schema.fields]
        except Exception:
            continue
    return None

present = {}
for t in META:
    cols = table_columns(t)
    if cols is not None:
        present[t] = cols
print("resolved tables:", sorted(present))

fk = {}
for (frm, fc, to, tc, _v) in PHYSICAL_RELS:
    fk.setdefault(frm, set()).add(fc); fk.setdefault(to, set()).add(tc)

node_rows = []
for t, cols in present.items():
    m = META[t]
    colmeta = []
    for (cn, ct) in cols:
        key = "pk" if PK.get(t) == cn else ("fk" if cn in fk.get(t, set()) else None)
        colmeta.append({"name": cn, "type": ct, "key": key})
    node_rows.append(Row(id=t, label=m["label"], table=t, category=m["category"], role=m.get("role","leaf"),
        grain=m["grain"], description=m["desc"], column_count=len(colmeta), columns_json=json.dumps(colmeta)))

node_ids = set(present)
seen = set(); edge_rows = []
def add_edge(frm, to, fc, tc, label, kind):
    if frm not in node_ids or to not in node_ids: return
    if (frm, to) in seen or (to, frm) in seen: return
    seen.add((frm, to))
    edge_rows.append(Row(id=f"{frm}::{to}", **{"from": frm, "to": to},
        from_col=fc, to_col=tc, label=label, kind=kind, cardinality="many-to-one"))

for (frm, fc, to, tc, verb) in PHYSICAL_RELS:
    add_edge(frm, to, fc, tc, verb, "physical")

# logical edges from shared keys
for t, cols in present.items():
    names = {c[0] for c in cols}
    if t != "dim_asset" and "asset_id" in names:
        add_edge(t, "dim_asset", "asset_id", "asset_id", "relates to", "logical")
    if t != "bridge_pi_tag_to_asset" and ("Tag" in names or "tag" in names):
        col = "Tag" if "Tag" in names else "tag"
        add_edge(t, "bridge_pi_tag_to_asset", col, "Tag", "relates to", "logical")

# temporal spine: connect date-bearing tables to dim_date
def date_col(names):
    for c in names:
        lc = c.lower()
        if lc in ("date_key","date"): return c
    for c in names:
        if c.lower().endswith("date"): return c
    return None
if "dim_date" in node_ids:
    for t, cols in present.items():
        if t in ("dim_date","dim_asset","dim_equipment","bridge_pi_tag_to_asset","selected_tags"): continue
        dc = date_col([c[0] for c in cols])
        if dc and (t, "dim_date") not in seen:
            seen.add((t, "dim_date"))
            edge_rows.append(Row(id=f"{t}::dim_date", **{"from": t, "to": "dim_date"},
                from_col=dc, to_col="date", label="dated by", kind="temporal", cardinality="many-to-one"))

print(f"nodes={len(node_rows)} edges={len(edge_rows)}")

In [ ]:
nodes_df = spark.createDataFrame(node_rows)
edges_df = spark.createDataFrame(edge_rows)

(nodes_df.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(f"{OUT_SCHEMA}.{NODES_TABLE}"))
(edges_df.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(f"{OUT_SCHEMA}.{EDGES_TABLE}"))

print(f"wrote {OUT_SCHEMA}.{NODES_TABLE} ({nodes_df.count()} rows) and {OUT_SCHEMA}.{EDGES_TABLE} ({edges_df.count()} rows)")
display(edges_df.orderBy("kind","from"))